# CD14+/CD16+ Mono and NK functional substates

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "fig3_S13a_S18_neurips2021_cite_bmmc_go_substates")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import textwrap
from matplotlib.colors import TwoSlopeNorm


In [ ]:
obodag = GODag(
    os.path.join(
        os.path.abspath("../.experiment_data/primary"), "preprocessing", "resources", "ebi_ensembl_go_mappings", "go-basic.obo"
    )
)


In [ ]:
(
    mdata,
    mdata_counts,
    adata,
    train_indices,
    val_indices,
    n_genes,
    n_regions,
    n_snps,
    labels_key,
    further_labels_keys,
    batch_key,
    patient_key,
    further_batch_keys,
    further_continuous_batch_keys,
    n_patient_covariates,
    protein_expression_obsm_key,
) = datasets.load_neurips2021_cite_BMMC(1.0, False, True)


In [ ]:
(
    mdata_unfilt,
    mdata_counts_filt,
    adata,
    train_indices,
    val_indices,
    n_genes,
    n_regions,
    n_snps,
    labels_key,
    further_labels_keys,
    batch_key,
    patient_key,
    further_batch_keys,
    further_continuous_batch_keys,
    n_patient_covariates,
    protein_expression_obsm_key,
) = datasets.load_neurips2021_cite_BMMC(1.0, False, False)


# Monocyte/DC substate marker-panel analysis (Fig. 3d, e)

In [ ]:
import umap
import scanpy as sc
import anndata as ad
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from scipy.stats import entropy


In [ ]:
resorted_mdata_counts = mdata_counts.mod["rna"][train_indices].copy()
resorted_mdata_counts_filt = mdata_counts_filt.mod["rna"][train_indices].copy()


In [ ]:
X_sparsevi = np.load(
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/X_sparsevi.npy"
)
X_sparsevi


In [ ]:
labels = mdata_counts.mod["rna"][train_indices].obs[labels_key]


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def get_ci_auroc(_graph_csv, go_term):
    activations = _graph_csv.loc[go_term]["activations_test"]
    case_activations = _graph_csv.loc[go_term]["case_activations"]

    n = len(activations)

    case_set = set(map(tuple, case_activations))
    labels = np.array([tuple(row) in case_set for row in activations], dtype=bool)

    X = activations
    y = labels.astype(int)

    clf = LogisticRegression(solver="liblinear", class_weight="balanced", max_iter=100)
    clf.fit(X, y)

    y_score = clf.predict_proba(X)[:, 1]

    auroc = roc_auc_score(y_true=y, y_score=y_score)

    def bootstrap_auc_ci(y_true, y_score, n_boot=200, alpha=0.05, random_state=42):
        rng = np.random.default_rng(random_state)
        y_true = np.asarray(y_true)
        y_score = np.asarray(y_score)
        n = len(y_true)
        boot_aucs = []
        for _ in range(n_boot):
            idx = rng.choice(n, size=n, replace=True)
            if len(np.unique(y_true[idx])) < 2:
                continue
            boot_aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
        lower = np.percentile(boot_aucs, 100 * alpha / 2)
        upper = np.percentile(boot_aucs, 100 * (1 - alpha / 2))
        median = np.median(boot_aucs)
        return median, lower, upper

    median, lower, upper = bootstrap_auc_ci(y, y_score)

    return auroc, median, lower, upper

def bootstrap_auc_ci(y_true, y_score, n_boot=200, alpha=0.05, random_state=42):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    n = len(y_true)
    boot_aucs = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    boot_aucs = np.array(boot_aucs)
    lower = np.percentile(boot_aucs, 100 * alpha / 2)
    upper = np.percentile(boot_aucs, 100 * (1 - alpha / 2))
    median = np.median(boot_aucs)
    return median, lower, upper, boot_aucs

import numpy as np
from scipy.stats import norm
from itertools import combinations

def delong_auc_variance(y_true, y_score):
    '''Computes AUC and its variance using the DeLong et al. (1988) method.'''
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    cases = y_score[y_true == 1]
    controls = y_score[y_true == 0]

    n_cases = len(cases)
    n_controls = len(controls)

    def placement_values(a, b):
        """For each a_i, compute mean of kernel(a_i, b_j) over all b_j."""
        result = np.zeros(len(a))
        for i, ai in enumerate(a):
            result[i] = np.mean((ai > b).astype(float) + 0.5 * (ai == b).astype(float))
        return result

    V_cases    = placement_values(cases, controls)
    V_controls = placement_values(controls, cases)

    auc = np.mean(V_cases)

    s10 = np.var(V_cases,    ddof=1) / n_cases
    s01 = np.var(V_controls, ddof=1) / n_controls

    variance = s10 + s01

    return auc, variance

def delong_test(y_true_1, y_score_1, y_true_2, y_score_2):
    '''Two-sided DeLong test comparing two AUROCs.'''
    y_true_1  = np.asarray(y_true_1)
    y_score_1 = np.asarray(y_score_1)
    y_true_2  = np.asarray(y_true_2)
    y_score_2 = np.asarray(y_score_2)

    same_samples = (len(y_true_1) == len(y_true_2)) and np.array_equal(y_true_1, y_true_2)

    auc1, var1 = delong_auc_variance(y_true_1, y_score_1)
    auc2, var2 = delong_auc_variance(y_true_2, y_score_2)

    if same_samples:
        cases_1    = y_score_1[y_true_1 == 1]
        controls_1 = y_score_1[y_true_1 == 0]
        cases_2    = y_score_2[y_true_2 == 1]
        controls_2 = y_score_2[y_true_2 == 0]

        n_cases    = len(cases_1)
        n_controls = len(controls_1)

        def placement_values(a, b):
            result = np.zeros(len(a))
            for i, ai in enumerate(a):
                result[i] = np.mean((ai > b).astype(float) + 0.5 * (ai == b).astype(float))
            return result

        V10_1 = placement_values(cases_1,    controls_1)
        V10_2 = placement_values(cases_2,    controls_2)
        V01_1 = placement_values(controls_1, cases_1)
        V01_2 = placement_values(controls_2, cases_2)

        cov10 = np.cov(V10_1, V10_2, ddof=1)[0, 1] / n_cases
        cov01 = np.cov(V01_1, V01_2, ddof=1)[0, 1] / n_controls

        variance_diff = var1 + var2 - 2 * (cov10 + cov01)
    else:
        variance_diff = var1 + var2

    if variance_diff <= 0:
        raise ValueError(
            f"Non-positive variance of AUC difference ({variance_diff:.6f}). "
            "Check that both groups have sufficient cases and controls."
        )

    z_stat = (auc1 - auc2) / np.sqrt(variance_diff)
    p_value = 2 * norm.sf(np.abs(z_stat))

    return auc1, auc2, z_stat, p_value


## GO Importance T antigen processing & presentation

In [ ]:
CD14_Mono = pd.read_pickle(
    f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_CD14+_Mono_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
)
CD16_Mono = pd.read_pickle(
    f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_CD16+_Mono_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
)


In [ ]:
CD14_Mono_Shuff = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_gshuff_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_CD14+ Mono_res_unp_act_mask_False_flip_False_genes_evaluated_graph_csv_with_ci.pkl"
)
CD16_Mono_Shuff = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_gshuff_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_CD16+ Mono_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv_with_ci.pkl"
)


In [ ]:
go_term = "GO:0002457"


In [ ]:
auroc, median, lower, upper = get_ci_auroc(CD14_Mono, go_term)
(abs(median - lower), abs(median - upper))


In [ ]:
auroc, median, lower, upper = get_ci_auroc(CD14_Mono_Shuff, go_term)
(abs(median - lower), abs(median - upper))


In [ ]:
from matplotlib.patches import Patch


In [ ]:
phenotype = "CD14+ Mono"
indices_phenotype = np.where(resorted_mdata_counts.obs[labels_key] == phenotype)[0]
print(len(indices_phenotype))
activations_phenotype_go_term = activations.loc[go_term]["activations_valid"][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)

indices_group_1 = np.where(
    (activations_phenotype_go_term.X.T[0] > -0.02) & (activations_phenotype_go_term.X.T[1] > -0.38)
)
indices_group_2 = np.where(
    (activations_phenotype_go_term.X.T[0] < -0.04) & (activations_phenotype_go_term.X.T[1] > -0.36)
)
indices_group_3 = np.where(
    (activations_phenotype_go_term.X.T[0] > 0.05)
    & (activations_phenotype_go_term.X.T[1] < -0.42)
    & (activations_phenotype_go_term.X.T[1] > -1)
)

c_phenotype = np.array(["#be9e7c"] * len(indices_phenotype))
c_phenotype[indices_group_1] = "#c9c9c9"
c_phenotype[indices_group_2] = "#70acc0"
c_phenotype[indices_group_3] = "#c12075"

plt.scatter(
    activations_phenotype_go_term.X.T[0],
    activations_phenotype_go_term.X.T[1],
    s=0.4,
    alpha=0.5,
    c=c_phenotype,
)
plt.grid(False)
label_names = [
    "Quiescent",
    "ICAM1-inflammatory",
    "Transitional-inflammatory",
    "Other CD14+ Monocytes",
]
colors_names = ["#c9c9c9", "#70acc0", "#c12075", "#be9e7c"]
legend_elements = [
    Patch(facecolor=colors_names[i], label=label_names[i]) for i in range(len(label_names))
]
plt.legend(handles=legend_elements, loc="best")
plt.tight_layout()
plt.show()


In [ ]:
auroc, median, lower, upper = get_ci_auroc(CD16_Mono, go_term)
(abs(median - lower), abs(median - upper))


## CD14+ Mono

In [ ]:
phenotype = "CD14+ Mono"
indices_phenotype = np.where(resorted_mdata_counts.obs[labels_key] == phenotype)[0]
print(len(indices_phenotype))
activations_phenotype_go_term = activations.loc[go_term]["activations_valid"][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)

plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.2)
plt.title(f"{phenotype} - {go_term} - {obodag[go_term].name}")
plt.show()
plt.close()


### Define indices & statistics

In [ ]:
indices_group_1 = np.where(
    (activations_phenotype_go_term.X.T[0] > -0.02) & (activations_phenotype_go_term.X.T[1] > -0.38)
)
indices_group_2 = np.where(
    (activations_phenotype_go_term.X.T[0] < -0.04) & (activations_phenotype_go_term.X.T[1] > -0.36)
)
indices_group_3 = np.where(
    (activations_phenotype_go_term.X.T[0] > 0.05)
    & (activations_phenotype_go_term.X.T[1] < -0.42)
    & (activations_phenotype_go_term.X.T[1] > -1)
)


### GO Importances for subgroups

In [ ]:
indices_group_x = indices_group_3
indices_subgroup = [
    list(resorted_mdata_counts.obs.index).index(cell)
    for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
]
print(len(list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup))
print(len(resorted_mdata_counts.obs))
indices_other_pheno_subgroup = (
    list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
)

activations_cd14_mono_group3 = activations.loc[go_term]["activations_valid"][indices_subgroup]


In [ ]:
max_iter = 100
results_per_group_cd14 = {}
_all_X = CD14_Mono.loc[go_term]["activations_test"]
_all_case = CD14_Mono.loc[go_term]["case_activations"]
_all_case_set = set(map(tuple, _all_case))
_all_y = np.array([tuple(row) in _all_case_set for row in _all_X], dtype=int)
_all_clf = LogisticRegression(solver="liblinear", class_weight="balanced", max_iter=100)
_all_clf.fit(_all_X, _all_y)
_all_preds = _all_clf.predict_proba(_all_X)[:, 1]
_all_roc = roc_auc_score(y_true=_all_y, y_score=_all_preds)
_all_median, _all_lower, _all_upper, _all_boot = bootstrap_auc_ci(_all_y, _all_preds)
print(f"[all] AUROC: {_all_roc:.4f} | CI: [{_all_lower:.4f}, {_all_upper:.4f}]")

results_per_group_cd14["all"] = {
    "y": _all_y, "preds": _all_preds, "auroc": _all_roc,
    "median": _all_median, "lower": _all_lower, "upper": _all_upper, "boot_aucs": _all_boot,
}

for group_name, indices_group_x in zip(
    ["group_1", "group_2", "group_3"], [indices_group_1, indices_group_2, indices_group_3],
):
    indices_subgroup = [
        list(resorted_mdata_counts.obs.index).index(cell)
        for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
    ]
    indices_other_pheno_subgroup = (
        list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
    )

    resorted_mdata_counts_filt_subgroup = resorted_mdata_counts[indices_other_pheno_subgroup]
    case_idces_valid = np.array(resorted_mdata_counts_filt_subgroup.obs[labels_key] == phenotype)

    X = activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup]
    y = case_idces_valid

    clf = LogisticRegression(solver="liblinear", max_iter=max_iter, class_weight="balanced")
    clf.fit(X, y)
    preds = clf.predict_proba(X)[:, 1]

    roc = roc_auc_score(y_true=y, y_score=preds)
    median, lower, upper, boot_aucs = bootstrap_auc_ci(y, preds)

    print(f"[{group_name}] AUROC: {roc:.4f} | CI: [{lower:.4f}, {upper:.4f}]")

    results_per_group_cd14[group_name] = {
        "y": y, "preds": preds, "auroc": roc,
        "median": median, "lower": lower, "upper": upper, "boot_aucs": boot_aucs,
    }

print("\nPairwise DeLong Tests (CD14)")
for g1, g2 in combinations(["group_1", "group_2", "group_3"], 2):
    auc1, auc2, z, p = delong_test(
        results_per_group_cd14[g1]["y"], results_per_group_cd14[g1]["preds"],
        results_per_group_cd14[g2]["y"], results_per_group_cd14[g2]["preds"],
    )
    results_per_group_cd14[f"{g1}_vs_{g2}_z"] = z
    results_per_group_cd14[f"{g1}_vs_{g2}_p"] = p
    print(f"[{g1} vs {g2}] AUC1={auc1:.4f}, AUC2={auc2:.4f}, Z={z:.4f}, p={p:.3e}")


In [ ]:
max_iter = 100

for indices_group_x in [indices_group_1, indices_group_2, indices_group_3]:
    indices_subgroup = [
        list(resorted_mdata_counts.obs.index).index(cell)
        for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
    ]
    print(
        len(
            list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
        )
    )
    print(len(resorted_mdata_counts.obs))
    indices_other_pheno_subgroup = (
        list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
    )
    resorted_mdata_counts_filt_subgroup = resorted_mdata_counts[indices_other_pheno_subgroup]
    case_idces_valid = np.array(resorted_mdata_counts_filt_subgroup.obs[labels_key] == phenotype)

    clf = LogisticRegression(solver="liblinear", max_iter=max_iter, class_weight="balanced")

    clf.fit(
        activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup],
        case_idces_valid,
    )

    preds = clf.predict_proba(
        activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup]
    )[:, 1]

    roc = roc_auc_score(y_true=case_idces_valid, y_score=preds)
    print(roc)


In [ ]:
max_iter = 100

for indices_group_x in [indices_group_1, indices_group_2, indices_group_3]:
    indices_subgroup = [
        list(resorted_mdata_counts.obs.index).index(cell)
        for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
    ]
    print(
        len(
            list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
        )
    )
    print(len(resorted_mdata_counts.obs))
    indices_other_pheno_subgroup = (
        list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
    )
    resorted_mdata_counts_filt_subgroup = resorted_mdata_counts[indices_other_pheno_subgroup]
    case_idces_valid = np.array(resorted_mdata_counts_filt_subgroup.obs[labels_key] == phenotype)

    clf = LogisticRegression(solver="liblinear", max_iter=max_iter, class_weight="balanced")

    clf.fit(
        activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup],
        case_idces_valid,
    )

    preds = clf.predict_proba(
        activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup]
    )[:, 1]

    roc = roc_auc_score(y_true=case_idces_valid, y_score=preds)
    print(roc)


### Plotting groups on UMAP

In [ ]:
indices_cd14_monocytes = np.where(labels == "CD14+ Mono")
indices_cd14_monocytes


In [ ]:
labels_filt = labels.iloc[indices_cd14_monocytes]
labels_filt


In [ ]:
labels_filt = labels_filt.astype("str")


In [ ]:
labels_filt[:] = "#be9e7c"
labels_filt


In [ ]:
labels_filt.loc[resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_1].index] = (
    "#c9c9c9"
)
labels_filt.loc[resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_2].index] = (
    "#70acc0"
)
labels_filt.loc[resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_3].index] = (
    "#c12075"
)


In [ ]:
sizes_filt = np.array([2.5] * len(labels_filt))
sizes_filt[np.where(labels_filt == "#c9c9c9")] = 0.5
sizes_filt


In [ ]:
X_sparsevi_filt = X_sparsevi[indices_cd14_monocytes]


In [ ]:
reducer = umap.UMAP(n_neighbors=15, n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_sparsevi_filt)


In [ ]:
import scanpy as sc

X_sparsevi_filt_adata = sc.AnnData(X=X_sparsevi_filt)
sc.pp.neighbors(X_sparsevi_filt_adata, n_neighbors=15, use_rep="X")


In [ ]:
sc.tl.umap(X_sparsevi_filt_adata)


In [ ]:
sc.tl.louvain(X_sparsevi_filt_adata, resolution=0.5)
X_sparsevi_filt_adata.obs["louvain"].value_counts()


In [ ]:
sc.pl.umap(X_sparsevi_filt_adata, color="louvain")


In [ ]:
X_umap = X_sparsevi_filt_adata.obsm["X_umap"]


In [ ]:
from matplotlib.patches import Patch
set_figure_params(4, 4)
cmap = ListedColormap(colors)
fig, ax = plt.subplots()
ax.set_aspect('equal', 'box')
sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=1.0, c=X_sparsevi_filt_adata.obs['louvain'].astype('int'), edgecolor='none')
ax.set_xticks([])
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_box_aspect(1)
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_louvain_{go_term}_{phenotype}_{mode}_sparsevi_umap_substates.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_louvain_{go_term}_{phenotype}_{mode}_sparsevi_umap_substates.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


In [ ]:
from matplotlib.patches import Patch
set_figure_params(4, 4)
cmap = ListedColormap(colors)
fig, ax = plt.subplots()
ax.set_aspect('equal', 'box')
sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=sizes_filt, c=labels_filt, cmap=cmap, edgecolor='none')
ax.set_xticks([])
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_box_aspect(1)
label_names = ['Quiescent', 'ICAM1-inflammatory', 'Transitional-inflammatory', 'Other CD14+ Monocytes']
colors_names = ['#c9c9c9', '#70acc0', '#c12075', '#be9e7c']
legend_elements = [Patch(facecolor=colors_names[i], label=label_names[i]) for i in range(len(label_names))]
ax.legend(handles=legend_elements, loc='best')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_sparsevi_umap_substates.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_sparsevi_umap_substates.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


### Plotting GO activation space

In [ ]:
resorted_mdata_counts_phenotype = resorted_mdata_counts[indices_phenotype].copy()
resorted_mdata_counts_filt_phenotype = resorted_mdata_counts_filt[indices_phenotype].copy()


In [ ]:
resorted_mdata_counts_phenotype.obs["louvain"] = np.array(
    X_sparsevi_filt_adata.obs["louvain"].astype("str")
)

import scanpy as sc

sc.pl.dotplot(
    resorted_mdata_counts_phenotype,
    ["CX3CR1", "CCL3", "IL1B", "FCGR3A", "ICAM1"],
    groupby="louvain",
)


In [ ]:
c_phenotype = np.array(["#be9e7c"] * len(indices_phenotype))
c_phenotype[indices_group_1] = "#c9c9c9"
c_phenotype[indices_group_2] = "#70acc0"
c_phenotype[indices_group_3] = "#c12075"


In [ ]:
c_batch = np.array(resorted_mdata_counts_phenotype.obs[batch_key].astype("str"))
c_batch[c_batch == "site1"] = "#0000FF"
c_batch[c_batch == "site2"] = "#ff0000"
c_batch[c_batch == "site3"] = "#008000"
c_batch[c_batch == "site4"] = "#FFFF00"


In [ ]:
set_figure_params(4, 4)
print(len(indices_phenotype))
activations_phenotype_go_term = activations.loc[go_term]['activations_valid'][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)
plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.4, alpha=0.5, c=c_batch)
plt.grid(False)
label_names = ['Site 1', 'Site 2', 'Site 3', 'Site 4']
colors_names = ['#0000FF', '#ff0000', '#008000', '#FFFF00']
legend_elements = [Patch(facecolor=colors_names[i], label=label_names[i]) for i in range(len(label_names))]
plt.legend(handles=legend_elements, title='Collection site', loc='best')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_activation_space_batches.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_activation_space_batches.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


In [ ]:
from matplotlib.patches import Patch


In [ ]:
set_figure_params(4, 4)
print(len(indices_phenotype))
activations_phenotype_go_term = CD14_Mono_Shuff.loc[go_term]['activations_valid'][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)
plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.4, alpha=0.5, c=c_phenotype)
plt.grid(False)
label_names = ['Quiescent', 'ICAM1-inflammatory', 'Transitional-inflammatory', 'Other CD14+ Monocytes']
colors_names = ['#c9c9c9', '#70acc0', '#c12075', '#be9e7c']
legend_elements = [Patch(facecolor=colors_names[i], label=label_names[i]) for i in range(len(label_names))]
plt.legend(handles=legend_elements, loc='best')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_gshuff_baseline_activation_space_substates.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_gshuff_baseline_activation_space_substates.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


In [ ]:
set_figure_params(4, 4)
print(len(indices_phenotype))
activations_phenotype_go_term = activations.loc[go_term]['activations_valid'][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)
plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.4, alpha=0.5, c=c_phenotype)
plt.grid(False)
label_names = ['Quiescent', 'ICAM1-inflammatory', 'Transitional-inflammatory', 'Other CD14+ Monocytes']
colors_names = ['#c9c9c9', '#70acc0', '#c12075', '#be9e7c']
legend_elements = [Patch(facecolor=colors_names[i], label=label_names[i]) for i in range(len(label_names))]
plt.legend(handles=legend_elements, loc='best')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_activation_space_substates.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_activation_space_substates.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


### Get dotplots

In [ ]:
resorted_mdata_counts_phenotype.obs[labels_key] = resorted_mdata_counts_phenotype.obs[
    labels_key
].astype("str")
resorted_mdata_counts_phenotype.obs[labels_key] = "Other CD14+ Monocytes"
resorted_mdata_counts_filt_phenotype.obs[labels_key] = resorted_mdata_counts_filt_phenotype.obs[
    labels_key
].astype("str")
resorted_mdata_counts_filt_phenotype.obs[labels_key] = "Other CD14+ Monocytes"
resorted_mdata_counts_phenotype.obs[labels_key].iloc[indices_group_1] = "Quiescent"
resorted_mdata_counts_phenotype.obs[labels_key].iloc[indices_group_2] = "ICAM1-inflammatory"
resorted_mdata_counts_phenotype.obs[labels_key].iloc[indices_group_3] = "Transitional-inflammatory"
resorted_mdata_counts_filt_phenotype.obs[labels_key].iloc[indices_group_1] = "Quiescent"
resorted_mdata_counts_filt_phenotype.obs[labels_key].iloc[indices_group_2] = "ICAM1-inflammatory"
resorted_mdata_counts_filt_phenotype.obs[labels_key].iloc[
    indices_group_3
] = "Transitional-inflammatory"


In [ ]:
resorted_mdata_counts_phenotype = resorted_mdata_counts_phenotype[
    np.where(resorted_mdata_counts_phenotype.obs[labels_key] != "Other CD14+ Monocytes")
].copy()
resorted_mdata_counts_phenotype = ad.concat(
    [resorted_mdata_counts_phenotype, resorted_mdata_counts[indices_phenotype]]
)


In [ ]:
resorted_mdata_counts_filt_phenotype = resorted_mdata_counts_filt_phenotype[
    np.where(resorted_mdata_counts_filt_phenotype.obs[labels_key] != "Other CD14+ Monocytes")
].copy()
resorted_mdata_counts_filt_phenotype = ad.concat(
    [resorted_mdata_counts_filt_phenotype, resorted_mdata_counts_filt[indices_phenotype]]
)


In [ ]:
resorted_mdata_counts_phenotype.obs[labels_key] = resorted_mdata_counts_phenotype.obs[
    labels_key
].replace({"CD14+ Mono": "All CD14+ Monocytes"})


In [ ]:
resorted_mdata_counts_filt_phenotype.obs[labels_key] = resorted_mdata_counts_filt_phenotype.obs[
    labels_key
].replace({"CD14+ Mono": "All CD14+ Monocytes"})


#### Actual dotplots

In [ ]:
import scanpy as sc


In [ ]:
_dp = sc.pl.dotplot(
    resorted_mdata_counts_phenotype,
    ["CX3CR1", "CCL3", "IL1B", "FCGR3A"],
    groupby=labels_key,
    categories_order=[
        "Quiescent",
        "ICAM1-inflammatory",
        "Transitional-inflammatory",
        "All CD14+ Monocytes",
    ],

    show=False,
    return_fig=True,
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_cd14_mono_inflammation_dotplot_CX3CR1_CCL3_IL1B_FCGR3A.png"),
    dpi=500, format="png", transparent=True, bbox_inches="tight",
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_cd14_mono_inflammation_dotplot_CX3CR1_CCL3_IL1B_FCGR3A.svg"),
    dpi=500, transparent=True, bbox_inches="tight", format="svg",
)
plt.show()


In [ ]:
_dp = sc.pl.dotplot(
    resorted_mdata_counts_phenotype,
    ["ICAM1"],
    groupby=labels_key,
    categories_order=[
        "Quiescent",
        "ICAM1-inflammatory",
        "Transitional-inflammatory",
        "All CD14+ Monocytes",
    ],

    show=False,
    return_fig=True,
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_cd14_mono_inflammation_dotplot_ICAM1.png"),
    dpi=500, format="png", transparent=True, bbox_inches="tight",
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_cd14_mono_inflammation_dotplot_ICAM1.svg"),
    dpi=500, transparent=True, bbox_inches="tight", format="svg",
)
plt.show()


## CD16+ Mono

In [ ]:
phenotype = "CD16+ Mono"
go_term = "GO:0002457"
indices_phenotype = np.where(resorted_mdata_counts.obs[labels_key] == phenotype)[0]
print(len(indices_phenotype))
activations_phenotype_go_term = activations.loc[go_term]["activations_valid"][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)
c_phenotype = np.array(["#000000"] * len(indices_phenotype))

plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.2)
plt.title(f"{phenotype} - {go_term} - {obodag[go_term].name}")
plt.show()
plt.close()


### Define indices & statistics

In [ ]:
indices_group_1 = np.where(
    (activations_phenotype_go_term.X.T[0] > 0) & (activations_phenotype_go_term.X.T[1] > -0.42)
)
indices_group_2 = np.where(
    (activations_phenotype_go_term.X.T[0] < 0) & (activations_phenotype_go_term.X.T[1] > -0.42)
)
indices_group_3 = np.where(
    (activations_phenotype_go_term.X.T[0] > 0.05)
    & (activations_phenotype_go_term.X.T[1] < -0.42)
    & (activations_phenotype_go_term.X.T[1] > -1)
)


### GO Importances for subgroups

In [ ]:
indices_group_x = indices_group_3
indices_subgroup = [
    list(resorted_mdata_counts.obs.index).index(cell)
    for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
]
print(len(list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup))
print(len(resorted_mdata_counts.obs))
indices_other_pheno_subgroup = (
    list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
)

activations_cd16_mono_group3 = activations.loc[go_term]["activations_valid"][indices_subgroup]


In [ ]:
max_iter = 100
results_per_group_cd16 = {}
_all_X = CD16_Mono.loc[go_term]["activations_test"]
_all_case = CD16_Mono.loc[go_term]["case_activations"]
_all_case_set = set(map(tuple, _all_case))
_all_y = np.array([tuple(row) in _all_case_set for row in _all_X], dtype=int)
_all_clf = LogisticRegression(solver="liblinear", class_weight="balanced", max_iter=100)
_all_clf.fit(_all_X, _all_y)
_all_preds = _all_clf.predict_proba(_all_X)[:, 1]
_all_roc = roc_auc_score(y_true=_all_y, y_score=_all_preds)
_all_median, _all_lower, _all_upper, _all_boot = bootstrap_auc_ci(_all_y, _all_preds)
print(f"[all] AUROC: {_all_roc:.4f} | CI: [{_all_lower:.4f}, {_all_upper:.4f}]")

results_per_group_cd16["all"] = {
    "y": _all_y, "preds": _all_preds, "auroc": _all_roc,
    "median": _all_median, "lower": _all_lower, "upper": _all_upper, "boot_aucs": _all_boot,
}

for group_name, indices_group_x in zip(
    ["group_1", "group_2", "group_3"], [indices_group_1, indices_group_2, indices_group_3],
):
    indices_subgroup = [
        list(resorted_mdata_counts.obs.index).index(cell)
        for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
    ]
    indices_other_pheno_subgroup = (
        list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
    )

    resorted_mdata_counts_filt_subgroup = resorted_mdata_counts[indices_other_pheno_subgroup]
    case_idces_valid = np.array(resorted_mdata_counts_filt_subgroup.obs[labels_key] == phenotype)

    X = activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup]
    y = case_idces_valid

    clf = LogisticRegression(solver="liblinear", max_iter=max_iter, class_weight="balanced")
    clf.fit(X, y)
    preds = clf.predict_proba(X)[:, 1]

    roc = roc_auc_score(y_true=y, y_score=preds)
    median, lower, upper, boot_aucs = bootstrap_auc_ci(y, preds)

    print(f"[{group_name}] AUROC: {roc:.4f} | CI: [{lower:.4f}, {upper:.4f}]")

    results_per_group_cd16[group_name] = {
        "y": y, "preds": preds, "auroc": roc,
        "median": median, "lower": lower, "upper": upper, "boot_aucs": boot_aucs,
    }

print("\nPairwise DeLong Tests (CD16)")
for g1, g2 in combinations(["group_1", "group_2", "group_3"], 2):
    auc1, auc2, z, p = delong_test(
        results_per_group_cd16[g1]["y"], results_per_group_cd16[g1]["preds"],
        results_per_group_cd16[g2]["y"], results_per_group_cd16[g2]["preds"],
    )
    results_per_group_cd16[f"{g1}_vs_{g2}_z"] = z
    results_per_group_cd16[f"{g1}_vs_{g2}_p"] = p
    print(f"[{g1} vs {g2}] AUC1={auc1:.4f}, AUC2={auc2:.4f}, Z={z:.4f}, p={p:.3e}")


In [ ]:
max_iter = 100

for indices_group_x in [indices_group_1, indices_group_2, indices_group_3]:
    indices_subgroup = [
        list(resorted_mdata_counts.obs.index).index(cell)
        for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
    ]
    print(
        len(
            list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
        )
    )
    print(len(resorted_mdata_counts.obs))
    indices_other_pheno_subgroup = (
        list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
    )
    resorted_mdata_counts_filt_subgroup = resorted_mdata_counts[indices_other_pheno_subgroup]
    case_idces_valid = np.array(resorted_mdata_counts_filt_subgroup.obs[labels_key] == phenotype)

    clf = LogisticRegression(solver="liblinear", max_iter=max_iter, class_weight="balanced")

    clf.fit(
        activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup],
        case_idces_valid,
    )

    preds = clf.predict_proba(
        activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup]
    )[:, 1]

    roc = roc_auc_score(y_true=case_idces_valid, y_score=preds)
    print(roc)


In [ ]:
max_iter = 100

for indices_group_x in [indices_group_1, indices_group_2, indices_group_3]:
    indices_subgroup = [
        list(resorted_mdata_counts.obs.index).index(cell)
        for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
    ]
    print(
        len(
            list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
        )
    )
    print(len(resorted_mdata_counts.obs))
    indices_other_pheno_subgroup = (
        list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
    )
    resorted_mdata_counts_filt_subgroup = resorted_mdata_counts[indices_other_pheno_subgroup]
    case_idces_valid = np.array(resorted_mdata_counts_filt_subgroup.obs[labels_key] == phenotype)

    clf = LogisticRegression(solver="liblinear", max_iter=max_iter, class_weight="balanced")

    clf.fit(
        activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup],
        case_idces_valid,
    )

    preds = clf.predict_proba(
        activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup]
    )[:, 1]

    roc = roc_auc_score(y_true=case_idces_valid, y_score=preds)
    print(roc)


### Plotting groups on UMAP

In [ ]:
indices_cd14_monocytes = np.where(labels == "CD16+ Mono")
indices_cd14_monocytes


In [ ]:
labels_filt = labels.iloc[indices_cd14_monocytes]
labels_filt


In [ ]:
labels_filt = labels_filt.astype("str")


In [ ]:
labels_filt[:] = "#be9e7c"
labels_filt


In [ ]:
labels_filt.loc[resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_1].index] = (
    "#c9c9c9"
)
labels_filt.loc[resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_2].index] = (
    "#70acc0"
)
labels_filt.loc[resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_3].index] = (
    "#c12075"
)


In [ ]:
X_sparsevi_filt = X_sparsevi[indices_cd14_monocytes]


In [ ]:
reducer = umap.UMAP(n_neighbors=15, n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_sparsevi_filt)


In [ ]:
from matplotlib.patches import Patch
set_figure_params(4, 4)
cmap = ListedColormap(colors)
fig, ax = plt.subplots()
ax.set_aspect('equal', 'box')
sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], s=0.7, c=labels_filt, cmap=cmap, edgecolor='none')
ax.set_xticks([])
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_box_aspect(1)
label_names = ['Quiescent', 'ICAM1-inflammatory', 'Transitional-inflammatory', 'Other CD16+ Monocytes']
colors_names = ['#c9c9c9', '#70acc0', '#c12075', '#be9e7c']
legend_elements = [Patch(facecolor=colors_names[i], label=label_names[i]) for i in range(len(label_names))]
ax.legend(handles=legend_elements, loc='best')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_sparsevi_umap_substates.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_sparsevi_umap_substates.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


### Plotting GO activation space

In [ ]:
resorted_mdata_counts_phenotype = resorted_mdata_counts[indices_phenotype].copy()
resorted_mdata_counts_filt_phenotype = resorted_mdata_counts_filt[indices_phenotype].copy()


In [ ]:
c_phenotype = np.array(["#be9e7c"] * len(indices_phenotype))
c_phenotype[indices_group_1] = "#c9c9c9"
c_phenotype[indices_group_2] = "#70acc0"
c_phenotype[indices_group_3] = "#c12075"


In [ ]:
c_batch = np.array(resorted_mdata_counts_phenotype.obs[batch_key].astype("str"))
c_batch[c_batch == "site1"] = "#0000FF"
c_batch[c_batch == "site2"] = "#ff0000"
c_batch[c_batch == "site3"] = "#008000"
c_batch[c_batch == "site4"] = "#FFFF00"


In [ ]:
set_figure_params(4, 4)
print(len(indices_phenotype))
activations_phenotype_go_term = activations.loc[go_term]['activations_valid'][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)
plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.4, alpha=0.5, c=c_batch)
plt.grid(False)
label_names = ['Site 1', 'Site 2', 'Site 3', 'Site 4']
colors_names = ['#0000FF', '#ff0000', '#008000', '#FFFF00']
legend_elements = [Patch(facecolor=colors_names[i], label=label_names[i]) for i in range(len(label_names))]
plt.legend(handles=legend_elements, title='Collection site', loc='best')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_activation_space_batches.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_activation_space_batches.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


In [ ]:
set_figure_params(4, 4)
print(len(indices_phenotype))
activations_phenotype_go_term = activations.loc[go_term]['activations_valid'][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)
plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.4, alpha=0.5, c=c_phenotype)
plt.grid(False)
label_names = ['Quiescent', 'ICAM1-inflammatory', 'Transitional-inflammatory', 'Other CD16+ Monocytes']
colors_names = ['#c9c9c9', '#70acc0', '#c12075', '#be9e7c']
legend_elements = [Patch(facecolor=colors_names[i], label=label_names[i]) for i in range(len(label_names))]
plt.legend(handles=legend_elements, loc='best')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_activation_space_substates.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_activation_space_substates.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


### Get dotplots

In [ ]:
resorted_mdata_counts_phenotype.obs[labels_key] = resorted_mdata_counts_phenotype.obs[
    labels_key
].astype("str")
resorted_mdata_counts_phenotype.obs[labels_key] = "Other CD14+ Monocytes"
resorted_mdata_counts_filt_phenotype.obs[labels_key] = resorted_mdata_counts_filt_phenotype.obs[
    labels_key
].astype("str")
resorted_mdata_counts_filt_phenotype.obs[labels_key] = "Other CD14+ Monocytes"
resorted_mdata_counts_phenotype.obs[labels_key].iloc[indices_group_1] = "Quiescent"
resorted_mdata_counts_phenotype.obs[labels_key].iloc[indices_group_2] = "ICAM1-inflammatory"
resorted_mdata_counts_phenotype.obs[labels_key].iloc[indices_group_3] = "Transitional-inflammatory"
resorted_mdata_counts_filt_phenotype.obs[labels_key].iloc[indices_group_1] = "Quiescent"
resorted_mdata_counts_filt_phenotype.obs[labels_key].iloc[indices_group_2] = "ICAM1-inflammatory"
resorted_mdata_counts_filt_phenotype.obs[labels_key].iloc[
    indices_group_3
] = "Transitional-inflammatory"


In [ ]:
resorted_mdata_counts_phenotype = resorted_mdata_counts_phenotype[
    np.where(resorted_mdata_counts_phenotype.obs[labels_key] != "Other CD14+ Monocytes")
].copy()
resorted_mdata_counts_phenotype = ad.concat(
    [resorted_mdata_counts_phenotype, resorted_mdata_counts[indices_phenotype]]
)


In [ ]:
resorted_mdata_counts_phenotype.obs[labels_key] = resorted_mdata_counts_phenotype.obs[
    labels_key
].replace({"CD16+ Mono": "All CD16+ Monocytes"})


#### Actual dotplots

In [ ]:
import scanpy as sc


In [ ]:
_dp = sc.pl.dotplot(
    resorted_mdata_counts_phenotype,
    ["CX3CR1", "CCL3", "IL1B"],
    groupby=labels_key,
    categories_order=[
        "Quiescent",
        "ICAM1-inflammatory",
        "Transitional-inflammatory",
        "All CD16+ Monocytes",
    ],

    show=False,
    return_fig=True,
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_cd16_mono_inflammation_dotplot_CX3CR1_CCL3_IL1B.png"),
    dpi=500, format="png", transparent=True, bbox_inches="tight",
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_cd16_mono_inflammation_dotplot_CX3CR1_CCL3_IL1B.svg"),
    dpi=500, transparent=True, bbox_inches="tight", format="svg",
)
plt.show()


In [ ]:
_dp = sc.pl.dotplot(
    resorted_mdata_counts_phenotype,
    ["CD14", "ICAM1"],
    groupby=labels_key,
    categories_order=[
        "Quiescent",
        "ICAM1-inflammatory",
        "Transitional-inflammatory",
        "All CD16+ Monocytes",
    ],

    show=False,
    return_fig=True,
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_cd16_mono_inflammation_dotplot_CD14_ICAM1.png"),
    dpi=500, format="png", transparent=True, bbox_inches="tight",
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_cd16_mono_inflammation_dotplot_CD14_ICAM1.svg"),
    dpi=500, transparent=True, bbox_inches="tight", format="svg",
)
plt.show()


## NK

In [ ]:
go_term = "GO:0002519"
phenotype = "NK"
indices_phenotype = np.where(resorted_mdata_counts.obs[labels_key] == phenotype)[0]
print(len(indices_phenotype))
activations_phenotype_go_term = activations.loc[go_term]["activations_valid"][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)

plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.2)
plt.title(f"{phenotype} - {go_term} - {obodag[go_term].name}")
plt.show()
plt.close()


### Define indices & statistics

In [ ]:
indices_group_1 = np.where(
    (activations_phenotype_go_term.X.T[0] < 0)
    & (activations_phenotype_go_term.X.T[0] > -0.8)
    & (activations_phenotype_go_term.X.T[1] < 0.35)
)
indices_group_2 = np.where(
    (activations_phenotype_go_term.X.T[0] > -0.5)
    & (activations_phenotype_go_term.X.T[1] > 0.30)
    & (activations_phenotype_go_term.X.T[1] < 0.75)
)
indices_group_3 = np.where(
    (activations_phenotype_go_term.X.T[0] < -1)
    & (activations_phenotype_go_term.X.T[1] > 0.2)
    & (activations_phenotype_go_term.X.T[1] < 0.55)
)
indices_group_4 = np.where(
    (activations_phenotype_go_term.X.T[0] < -0.9)
    & (activations_phenotype_go_term.X.T[0] > -1.8)
    & (activations_phenotype_go_term.X.T[1] > 0.5)
    & (activations_phenotype_go_term.X.T[1] < 0.9)
)


In [ ]:
indices_group_4 = np.where(
    (activations_phenotype_go_term.X.T[0] < 0)
    & (activations_phenotype_go_term.X.T[0] > -0.8)
    & (activations_phenotype_go_term.X.T[1] < 0.35)
)
indices_group_3 = np.where(
    (activations_phenotype_go_term.X.T[0] > -0.5)
    & (activations_phenotype_go_term.X.T[1] > 0.30)
    & (activations_phenotype_go_term.X.T[1] < 0.75)
)
indices_group_2 = np.where(
    (activations_phenotype_go_term.X.T[0] < -1)
    & (activations_phenotype_go_term.X.T[1] > 0.2)
    & (activations_phenotype_go_term.X.T[1] < 0.55)
)
indices_group_1 = np.where(
    (activations_phenotype_go_term.X.T[0] < -0.9)
    & (activations_phenotype_go_term.X.T[0] > -1.8)
    & (activations_phenotype_go_term.X.T[1] > 0.5)
    & (activations_phenotype_go_term.X.T[1] < 0.9)
)


### GO Importances for subgroups

In [ ]:
indices_group_x = indices_group_3
indices_subgroup = [
    list(resorted_mdata_counts.obs.index).index(cell)
    for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
]
print(len(list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup))
print(len(resorted_mdata_counts.obs))
indices_other_pheno_subgroup = (
    list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
)

activations_naive_cd20_b_igkcpos_group3 = activations.loc[go_term]["activations_valid"][
    indices_subgroup
]


In [ ]:
max_iter = 100

for indices_group_x in [indices_group_1, indices_group_2, indices_group_3, indices_group_4]:
    indices_subgroup = [
        list(resorted_mdata_counts.obs.index).index(cell)
        for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
    ]
    print(
        len(
            list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
        )
    )
    print(len(resorted_mdata_counts.obs))
    indices_other_pheno_subgroup = (
        list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
    )
    resorted_mdata_counts_filt_subgroup = resorted_mdata_counts[indices_other_pheno_subgroup]
    case_idces_valid = np.array(resorted_mdata_counts_filt_subgroup.obs[labels_key] == phenotype)

    clf = LogisticRegression(solver="liblinear", max_iter=max_iter, class_weight="balanced")

    clf.fit(
        activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup],
        case_idces_valid,
    )

    preds = clf.predict_proba(
        activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup]
    )[:, 1]

    roc = roc_auc_score(y_true=case_idces_valid, y_score=preds)
    print(roc)


### Plotting groups on UMAP

In [ ]:
indices_cd14_monocytes = np.where(labels == "NK")
indices_cd14_monocytes


In [ ]:
labels_filt = labels.iloc[indices_cd14_monocytes]
labels_filt


In [ ]:
labels_filt = labels_filt.astype("str")


In [ ]:
labels_filt[:] = "#000000"
labels_filt


In [ ]:
labels_filt.loc[resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_1].index] = (
    "#e57373"
)
labels_filt.loc[resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_2].index] = (
    "#003754"
)
labels_filt.loc[resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_3].index] = (
    "#d4d406"
)
labels_filt.loc[resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_4].index] = (
    "#00695c"
)


In [ ]:
X_sparsevi_filt = X_sparsevi[indices_cd14_monocytes]


In [ ]:
reducer = umap.UMAP(n_neighbors=15, n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_sparsevi_filt)


### Plotting GO activation space

In [ ]:
resorted_mdata_counts_phenotype = resorted_mdata_counts[indices_phenotype].copy()
resorted_mdata_counts_filt_phenotype = resorted_mdata_counts_filt[indices_phenotype].copy()


In [ ]:
c_phenotype = np.array(["#000000"] * len(indices_phenotype))
c_phenotype[indices_group_1] = "#e57373"
c_phenotype[indices_group_2] = "#003754"
c_phenotype[indices_group_3] = "#d4d406"
c_phenotype[indices_group_4] = "#00695c"


In [ ]:
c_batch = np.array(resorted_mdata_counts_phenotype.obs[batch_key].astype("str"))
c_batch[c_batch == "site1"] = "#0000FF"
c_batch[c_batch == "site2"] = "#ff0000"
c_batch[c_batch == "site3"] = "#008000"
c_batch[c_batch == "site4"] = "#FFFF00"


In [ ]:
set_figure_params(4, 4)
print(len(indices_phenotype))
activations_phenotype_go_term = activations.loc[go_term]['activations_valid'][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)
plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.4, alpha=0.5, c=c_phenotype)
plt.grid(False)
label_names = ['CD56$^{\\mathrm{bright}}$ NK + Transitional NK', 'Active NK |', 'Active NK ||', 'Mature', 'Other NK']
colors_names = ['#e57373', '#003754', '#d4d406', '#00695c', '#000000']
legend_elements = [Patch(facecolor=colors_names[i], label=label_names[i]) for i in range(len(label_names))]
plt.legend(handles=legend_elements, loc='best')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_activation_space_substates.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, f'go_substates_{go_term}_{phenotype}_{mode}_activation_space_substates.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


### Get dotplots

In [ ]:
resorted_mdata_counts_phenotype.obs[labels_key] = resorted_mdata_counts_phenotype.obs[
    labels_key
].astype("str")
resorted_mdata_counts_phenotype.obs[labels_key] = "Other NK"
resorted_mdata_counts_filt_phenotype.obs[labels_key] = resorted_mdata_counts_filt_phenotype.obs[
    labels_key
].astype("str")
resorted_mdata_counts_filt_phenotype.obs[labels_key] = "Other NK"
resorted_mdata_counts_phenotype.obs[labels_key].iloc[
    indices_group_1
] = "CD56$^{\\mathrm{bright}}$ NK + Transitional NK"
resorted_mdata_counts_phenotype.obs[labels_key].iloc[indices_group_2] = "Active NK |"
resorted_mdata_counts_phenotype.obs[labels_key].iloc[indices_group_3] = "Active NK ||"
resorted_mdata_counts_phenotype.obs[labels_key].iloc[indices_group_4] = "Mature"
resorted_mdata_counts_filt_phenotype.obs[labels_key].iloc[
    indices_group_1
] = "CD56$^{\\mathrm{bright}}$ NK + Transitional NK"
resorted_mdata_counts_filt_phenotype.obs[labels_key].iloc[indices_group_2] = "Active NK |"
resorted_mdata_counts_filt_phenotype.obs[labels_key].iloc[indices_group_3] = "Active NK ||"
resorted_mdata_counts_filt_phenotype.obs[labels_key].iloc[indices_group_4] = "Mature"


In [ ]:
resorted_mdata_counts_phenotype = resorted_mdata_counts_phenotype[
    np.where(resorted_mdata_counts_phenotype.obs[labels_key] != "Other NK")
].copy()
resorted_mdata_counts_phenotype = ad.concat(
    [resorted_mdata_counts_phenotype, resorted_mdata_counts[indices_phenotype]]
)


In [ ]:
resorted_mdata_counts_filt_phenotype = resorted_mdata_counts_filt_phenotype[
    np.where(resorted_mdata_counts_filt_phenotype.obs[labels_key] != "Other NK")
].copy()
resorted_mdata_counts_filt_phenotype = ad.concat(
    [resorted_mdata_counts_filt_phenotype, resorted_mdata_counts_filt[indices_phenotype]]
)


In [ ]:
resorted_mdata_counts_phenotype.obs[labels_key] = resorted_mdata_counts_phenotype.obs[
    labels_key
].replace({"NK": "All NK"})


In [ ]:
resorted_mdata_counts_filt_phenotype.obs[labels_key] = resorted_mdata_counts_filt_phenotype.obs[
    labels_key
].replace({"NK": "All NK"})


In [ ]:
import scanpy as sc


In [ ]:
_dp = sc.pl.dotplot(
    resorted_mdata_counts_filt_phenotype,
    [
        "NCAM1",
        "TIGIT",
        "PTPN6",
        "SELL",
    ],
    groupby=labels_key,
    categories_order=[
        "CD56$^{\\mathrm{bright}}$ NK + Transitional NK",
        "Active NK |",
        "Active NK ||",
        "Mature",
        "All NK",
    ],
    show=False,
    return_fig=True,
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_nk_substates_dotplot_NCAM1_TIGIT_PTPN6_SELL.png"),
    dpi=500, format="png", transparent=True, bbox_inches="tight",
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_nk_substates_dotplot_NCAM1_TIGIT_PTPN6_SELL.svg"),
    dpi=500, transparent=True, bbox_inches="tight", format="svg",
)
plt.show()


In [ ]:
_dp = sc.pl.dotplot(
    resorted_mdata_counts_filt_phenotype,
    ["HAVCR2", "FCGR3A"],
    groupby=labels_key,
    categories_order=[
        "CD56$^{\\mathrm{bright}}$ NK + Transitional NK",
        "Active NK |",
        "Active NK ||",
        "Mature",
        "All NK",
    ],
    show=False,
    return_fig=True,
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_nk_substates_dotplot_HAVCR2_FCGR3A.png"),
    dpi=500, format="png", transparent=True, bbox_inches="tight",
)
_dp.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_nk_substates_dotplot_HAVCR2_FCGR3A.svg"),
    dpi=500, transparent=True, bbox_inches="tight", format="svg",
)
plt.show()


## GO importance for CD14+/CD16+ Monocyte substates (Fig. 3c)

In [ ]:
go_term = "GO:0002457"

def add_significance_bracket(ax, x1, x2, y, p_value, h=0.01):
    if p_value < 0.001:
        sig = "***"
    elif p_value < 0.01:
        sig = "**"
    elif p_value < 0.05:
        sig = "*"
    else:
        sig = "n.s."

    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=0.8, color="black")
    ax.text((x1 + x2) / 2, y + h * 0.5, sig, ha="center", va="bottom", fontsize=6)


In [ ]:
results = results_per_group_cd14
cd14_values = [results["all"]["median"], results["group_1"]["median"], results["group_2"]["median"], results["group_3"]["median"]]
cd14_ci_lower = [results["all"]["lower"], results["group_1"]["lower"], results["group_2"]["lower"], results["group_3"]["lower"]]
cd14_ci_upper = [results["all"]["upper"], results["group_1"]["upper"], results["group_2"]["upper"], results["group_3"]["upper"]]
cd14_boot = [results["all"]["boot_aucs"], results["group_1"]["boot_aucs"], results["group_2"]["boot_aucs"], results["group_3"]["boot_aucs"]]

cd14_yerr_lower = [abs(cd14_values[i] - cd14_ci_lower[i]) for i in range(len(cd14_values))]
cd14_yerr_upper = [abs(cd14_ci_upper[i] - cd14_values[i]) for i in range(len(cd14_values))]

sub_labels_cd14 = ["All CD14+\nMonocytes", "Quiescent", "ICAM1-\ninflammatory", "Transitional-\ninflammatory"]
colors = ["#000000", "#c9c9c9", "#70acc0", "#c12075", "#000000", "#c9c9c9", "#70acc0", "#c12075"]
x_positions = [0, 1.2, 2.2, 3.2]

set_figure_params(1.3, 2.3)
fig, ax = plt.subplots(1, 1)
ax.grid(False)

bars = ax.bar(
    x_positions, cd14_values,
    color=colors[:4],
    yerr=[cd14_yerr_lower, cd14_yerr_upper],
    error_kw=dict(elinewidth=0.8, ecolor="#666666", capsize=2),
)

rng_jitter = np.random.default_rng(0)
for xpos, boot in zip(x_positions, cd14_boot):
    jitter = rng_jitter.uniform(-0.18, 0.18, size=len(boot))
    ax.scatter(xpos + jitter, boot, s=1.5, color="black", alpha=0.25, zorder=3, linewidths=0)

ax.set_ylim(0.45, 1.09)
ax.set_xticks(x_positions)
ax.set_xticklabels(sub_labels_cd14, rotation=90)
ax.set_ylabel("GO importance")
ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=0.6)

p_g1_g2 = results["group_1_vs_group_2_p"]
p_g1_g3 = results["group_1_vs_group_3_p"]
z_g1_g2 = results["group_1_vs_group_2_z"]
z_g1_g3 = results["group_1_vs_group_3_z"]
add_significance_bracket(ax, x_positions[1], x_positions[2], y=1.005, p_value=p_g1_g2, h=0.004)
add_significance_bracket(ax, x_positions[1], x_positions[3], y=1.045, p_value=p_g1_g3, h=0.004)
print("CD14 Quiescent vs ICAM1-inflammatory: z=%.4f, p=%.3e" % (z_g1_g2, p_g1_g2))
print("CD14 Quiescent vs Transitional-inflammatory: z=%.4f, p=%.3e" % (z_g1_g3, p_g1_g3))
print("n_bootstrap = %d per group" % len(cd14_boot[0]))

plt.tight_layout()
plt.savefig(
    os.path.join(_OUTDIR_ABS, f"go_substates_{go_term}_{mode}_cd14_mono_go_importances_substates.png"),
    dpi=500, format="png", transparent=True, bbox_inches="tight",
)
plt.savefig(
    os.path.join(_OUTDIR_ABS, f"go_substates_{go_term}_{mode}_cd14_mono_go_importances_substates.svg"),
    dpi=500, transparent=True, bbox_inches="tight", format="svg",
)


In [ ]:
results = results_per_group_cd16
cd16_values = [results["all"]["median"], results["group_1"]["median"], results["group_2"]["median"], results["group_3"]["median"]]
cd16_ci_lower = [results["all"]["lower"], results["group_1"]["lower"], results["group_2"]["lower"], results["group_3"]["lower"]]
cd16_ci_upper = [results["all"]["upper"], results["group_1"]["upper"], results["group_2"]["upper"], results["group_3"]["upper"]]
cd16_boot = [results["all"]["boot_aucs"], results["group_1"]["boot_aucs"], results["group_2"]["boot_aucs"], results["group_3"]["boot_aucs"]]

cd16_yerr_lower = [abs(cd16_values[i] - cd16_ci_lower[i]) for i in range(len(cd16_values))]
cd16_yerr_upper = [abs(cd16_ci_upper[i] - cd16_values[i]) for i in range(len(cd16_values))]

sub_labels_cd16 = ["All CD16+\nMonocytes", "Quiescent", "ICAM1-\ninflammatory", "Transitional-\ninflammatory"]
x_positions = [0, 1.4, 2.4, 3.4]

set_figure_params(1.3, 2.3)
fig, ax = plt.subplots(1, 1)
ax.grid(False)

bars = ax.bar(
    x_positions, cd16_values,
    color=colors[4:],
    yerr=[cd16_yerr_lower, cd16_yerr_upper],
    error_kw=dict(elinewidth=0.8, ecolor="#666666", capsize=2),
)

rng_jitter = np.random.default_rng(0)
for xpos, boot in zip(x_positions, cd16_boot):
    jitter = rng_jitter.uniform(-0.18, 0.18, size=len(boot))
    ax.scatter(xpos + jitter, boot, s=1.5, color="black", alpha=0.25, zorder=3, linewidths=0)

ax.set_ylim(0.45, 1.09)
ax.set_xticks(x_positions)
ax.set_xticklabels(sub_labels_cd16, rotation=90)
ax.set_ylabel("GO importance")
ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=0.6)

p_g1_g2 = results["group_1_vs_group_2_p"]
p_g1_g3 = results["group_1_vs_group_3_p"]
z_g1_g2 = results["group_1_vs_group_2_z"]
z_g1_g3 = results["group_1_vs_group_3_z"]
add_significance_bracket(ax, x_positions[1], x_positions[2], y=1.005, p_value=p_g1_g2, h=0.008)
add_significance_bracket(ax, x_positions[1], x_positions[3], y=1.045, p_value=p_g1_g3, h=0.008)
print("CD16 Quiescent vs ICAM1-inflammatory: z=%.4f, p=%.3e" % (z_g1_g2, p_g1_g2))
print("CD16 Quiescent vs Transitional-inflammatory: z=%.4f, p=%.3e" % (z_g1_g3, p_g1_g3))
print("n_bootstrap = %d per group" % len(cd16_boot[0]))

plt.tight_layout()
plt.savefig(
    os.path.join(_OUTDIR_ABS, f"go_substates_{go_term}_{mode}_cd16_mono_go_importances_substates.png"),
    dpi=500, format="png", transparent=True, bbox_inches="tight",
)
plt.savefig(
    os.path.join(_OUTDIR_ABS, f"go_substates_{go_term}_{mode}_cd16_mono_go_importances_substates.svg"),
    dpi=500, transparent=True, bbox_inches="tight", format="svg",
)


## Superseded (duplicate, unused)

In [ ]:
max_iter = 100
results_per_group = {}

for group_name, indices_group_x in zip(
    ["group_1", "group_2", "group_3"], [indices_group_1, indices_group_2, indices_group_3]
):
    indices_subgroup = [
        list(resorted_mdata_counts.obs.index).index(cell)
        for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
    ]
    indices_other_pheno_subgroup = (
        list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
    )

    resorted_mdata_counts_filt_subgroup = resorted_mdata_counts[indices_other_pheno_subgroup]
    case_idces_valid = np.array(resorted_mdata_counts_filt_subgroup.obs[labels_key] == phenotype)

    X = activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup]
    y = case_idces_valid

    clf = LogisticRegression(solver="liblinear", max_iter=max_iter, class_weight="balanced")
    clf.fit(X, y)
    preds = clf.predict_proba(X)[:, 1]

    roc = roc_auc_score(y_true=y, y_score=preds)
    median, lower, upper, boot_aucs = bootstrap_auc_ci(y, preds)

    print(f"[{group_name}] AUROC: {roc:.4f} | CI: [{lower:.4f}, {upper:.4f}]")

    results_per_group[group_name] = {"y": y, "preds": preds}

print("\nPairwise DeLong Tests")
group_names = list(results_per_group.keys())
for g1, g2 in combinations(group_names, 2):
    auc1, auc2, z, p = delong_test(
        results_per_group[g1]["y"],
        results_per_group[g1]["preds"],
        results_per_group[g2]["y"],
        results_per_group[g2]["preds"],
    )
    print(f"[{g1} vs {g2}] AUC1={auc1:.4f}, AUC2={auc2:.4f}, Z={z:.4f}, p={p:.4f}")


In [ ]:
max_iter = 100
results_per_group = {}

for group_name, indices_group_x in zip(
    ["group_1", "group_2", "group_3"], [indices_group_1, indices_group_2, indices_group_3]
):
    indices_subgroup = [
        list(resorted_mdata_counts.obs.index).index(cell)
        for cell in tqdm(resorted_mdata_counts[indices_phenotype].obs.iloc[indices_group_x].index)
    ]
    indices_other_pheno_subgroup = (
        list(np.where(resorted_mdata_counts.obs[labels_key] != phenotype)[0]) + indices_subgroup
    )

    resorted_mdata_counts_filt_subgroup = resorted_mdata_counts[indices_other_pheno_subgroup]
    case_idces_valid = np.array(resorted_mdata_counts_filt_subgroup.obs[labels_key] == phenotype)

    X = activations.loc[go_term]["activations_valid"][indices_other_pheno_subgroup]
    y = case_idces_valid

    clf = LogisticRegression(solver="liblinear", max_iter=max_iter, class_weight="balanced")
    clf.fit(X, y)
    preds = clf.predict_proba(X)[:, 1]

    roc = roc_auc_score(y_true=y, y_score=preds)
    median, lower, upper, boot_aucs = bootstrap_auc_ci(y, preds)

    print(f"[{group_name}] AUROC: {roc:.4f} | CI: [{lower:.4f}, {upper:.4f}]")

    results_per_group[group_name] = {"y": y, "preds": preds}

print("\nPairwise DeLong Tests")
group_names = list(results_per_group.keys())
for g1, g2 in combinations(group_names, 2):
    auc1, auc2, z, p = delong_test(
        results_per_group[g1]["y"],
        results_per_group[g1]["preds"],
        results_per_group[g2]["y"],
        results_per_group[g2]["preds"],
    )
    print(f"[{g1} vs {g2}] AUC1={auc1:.4f}, AUC2={auc2:.4f}, Z={z:.4f}, p={p:.4f}")


## Superseded (not used in the manuscript): cDC1 marker setup, cross-seed/shuffle activation-space scatter plots, unrelated B-cell checks

## Setup: DC marker gene panel (`genes_of_interest`)

In [ ]:
panglaodb = pd.read_csv(
    "../.experiment_data/primary/PanglaoDB_markers_27_Mar_2020.tsv", delimiter="\t"
)
panglaodb


In [ ]:
import pandas as pd

_GENE_MAPPING_PATH = os.path.abspath(
    os.path.join(
        "..", "preprocessing", "resources", "ensembl_gene_id_gene_name_mapping",
        "ensembl_gene_id_gene_name_mapping.csv",
    )
)

def _get_ensembl_gene_id_gene_name_mapping():
    return pd.read_csv(_GENE_MAPPING_PATH)


In [ ]:
ensembl_gene_id_gene_name_mapping = _get_ensembl_gene_id_gene_name_mapping()


In [ ]:
all_ensembl_gene_ids = []
for gene in np.unique(
    panglaodb[panglaodb["cell type"] == "Dendritic cells"]["official gene symbol"]
):
    ensembl_gene_ids = list(
        ensembl_gene_id_gene_name_mapping[ensembl_gene_id_gene_name_mapping["Gene name"] == gene][
            "Gene stable ID"
        ]
    )
    all_ensembl_gene_ids += ensembl_gene_ids


In [ ]:
cDC1 = pd.read_pickle(
    f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_cDC1_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
)
activations = cDC1


In [ ]:
CD14_Mono_Shuff = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_gshuff_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_CD14+ Mono_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv_with_ci.pkl"
)
CD16_Mono_Shuff = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_gshuff_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_CD16+ Mono_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv_with_ci.pkl"
)


In [ ]:
CD14_Mono_s1 = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s1/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_CD14+ Mono_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv_with_ci.pkl"
)
CD16_Mono_s1 = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s1/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_CD14+ Mono_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv_with_ci.pkl"
)
CD14_Mono_s2 = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s2/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_CD14+ Mono_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv_with_ci.pkl"
)
CD16_Mono_s2 = pd.read_pickle(
    "../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s2/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_CD14+ Mono_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv_with_ci.pkl"
)


In [ ]:
phenotype = "CD14+ Mono"
indices_phenotype = np.where(resorted_mdata_counts.obs[labels_key] == phenotype)[0]
print(len(indices_phenotype))
activations_phenotype_go_term = CD14_Mono_s1.loc[go_term]["activations_valid"][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)

plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.2)
plt.title(f"{phenotype} - {go_term} - {obodag[go_term].name}")
plt.show()
plt.close()


In [ ]:
phenotype = "CD14+ Mono"
indices_phenotype = np.where(resorted_mdata_counts.obs[labels_key] == phenotype)[0]
print(len(indices_phenotype))
activations_phenotype_go_term = CD14_Mono_s1.loc[go_term]["activations_valid"][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)

plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.2)
plt.title(f"{phenotype} - {go_term} - {obodag[go_term].name}")
plt.show()
plt.close()
indices_group_1_s1 = np.where(
    (activations_phenotype_go_term.X.T[0] > -0.04)
    & (activations_phenotype_go_term.X.T[1] > -0.3)
    & (activations_phenotype_go_term.X.T[1] < 0.35)
)
indices_group_2_s1 = np.where(
    (activations_phenotype_go_term.X.T[0] < -0.04) & (activations_phenotype_go_term.X.T[1] < 0.03)
)
indices_group_3_s1 = np.where(
    (activations_phenotype_go_term.X.T[0] < 0.03)
    & (activations_phenotype_go_term.X.T[0] > -0.03)
    & (activations_phenotype_go_term.X.T[1] > 0.15)
)


In [ ]:
phenotype = "CD14+ Mono"
indices_phenotype = np.where(resorted_mdata_counts.obs[labels_key] == phenotype)[0]
print(len(indices_phenotype))
activations_phenotype_go_term = CD14_Mono_s1.loc[go_term]["activations_valid"][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)

indices_group_1_s1 = np.where(
    (activations_phenotype_go_term.X.T[0] > -0.34)
    & (activations_phenotype_go_term.X.T[1] > -0.3)
    & (activations_phenotype_go_term.X.T[1] < 0.35)
)
indices_group_2_s1 = np.where(
    (activations_phenotype_go_term.X.T[0] < -0.34) & (activations_phenotype_go_term.X.T[1] < 0.03)
)
indices_group_3_s1 = np.where(
    (activations_phenotype_go_term.X.T[0] < 0.12)
    & (activations_phenotype_go_term.X.T[0] > -0.08)
    & (activations_phenotype_go_term.X.T[1] > 0.15)
)

c_phenotype = np.array(["#be9e7c"] * len(indices_phenotype))
c_phenotype[indices_group_1_s1] = "#c9c9c9"
c_phenotype[indices_group_2_s1] = "#70acc0"
c_phenotype[indices_group_3_s1] = "#c12075"

plt.scatter(
    activations_phenotype_go_term.X.T[0],
    activations_phenotype_go_term.X.T[1],
    s=0.4,
    alpha=0.5,
    c=c_phenotype,
)
plt.grid(False)
label_names = [
    "Quiescent",
    "ICAM1-inflammatory",
    "Transitional-inflammatory",
    "Other CD14+ Monocytes",
]
colors_names = ["#c9c9c9", "#70acc0", "#c12075", "#be9e7c"]
legend_elements = [
    Patch(facecolor=colors_names[i], label=label_names[i]) for i in range(len(label_names))
]
plt.legend(handles=legend_elements, loc="best")
plt.tight_layout()
plt.show()


In [ ]:
phenotype = "CD16+ Mono"
indices_phenotype = np.where(resorted_mdata_counts.obs[labels_key] == phenotype)[0]
print(len(indices_phenotype))
activations_phenotype_go_term = CD16_Mono_s1.loc[go_term]["activations_valid"][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)

plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.2)
plt.title(f"{phenotype} - {go_term} - {obodag[go_term].name}")
plt.show()
plt.close()


In [ ]:
phenotype = "CD14+ Mono"
indices_phenotype = np.where(resorted_mdata_counts.obs[labels_key] == phenotype)[0]
print(len(indices_phenotype))
activations_phenotype_go_term = CD14_Mono_s2.loc[go_term]["activations_valid"][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)

plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.2)
plt.title(f"{phenotype} - {go_term} - {obodag[go_term].name}")
plt.show()
plt.close()


In [ ]:
phenotype = "CD16+ Mono"
indices_phenotype = np.where(resorted_mdata_counts.obs[labels_key] == phenotype)[0]
print(len(indices_phenotype))
activations_phenotype_go_term = CD16_Mono_s2.loc[go_term]["activations_valid"][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)

plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.2)
plt.title(f"{phenotype} - {go_term} - {obodag[go_term].name}")
plt.show()
plt.close()


In [ ]:
phenotype = "CD14+ Mono"
indices_phenotype = np.where(resorted_mdata_counts.obs[labels_key] == phenotype)[0]
print(len(indices_phenotype))
activations_phenotype_go_term = CD14_Mono_Shuff.loc[go_term]["activations_valid"][indices_phenotype]
activations_phenotype_go_term = ad.AnnData(activations_phenotype_go_term)

plt.scatter(activations_phenotype_go_term.X.T[0], activations_phenotype_go_term.X.T[1], s=0.2)
plt.title(f"{phenotype} - {go_term} - {obodag[go_term].name}")
plt.show()
plt.close()


In [ ]:
auroc, median, lower, upper = get_ci_auroc(Naive_CD20_B_IGKCpos, go_term)
(abs(median - lower), abs(median - upper))


In [ ]:
auroc, median, lower, upper = get_ci_auroc(Naive_CD20_B_IGKCneg, go_term)
(abs(median - lower), abs(median - upper))
